# g4 E4B LoRA Trainer (exp12_e4b_lora)

In [ ]:
import os, sys, json, glob, re, random, time, base64
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("python", sys.version)
for m in ["torch","transformers","peft","accelerate","datasets"]:
    try:
        mod=__import__(m); print(m, "ok", getattr(mod,"__version__",""))
    except Exception as e:
        print(m, "MISSING", e)


In [ ]:
def find_tasks():
    cands = [
        os.environ.get("TASKS_JSONL"),
        "/kaggle/input/competitions/gemma-4-developer-agent/published/tasks.jsonl",
        "/kaggle/input/gemma-4-developer-agent/published/tasks.jsonl",
        "/kaggle/input/gemma-4-developer-agent/tasks.jsonl",
        "/kaggle/working/published/tasks.jsonl",
        "/kaggle/working/tasks.jsonl",
    ]
    for c in cands:
        if c and os.path.isfile(c):
            return c
    for root,_,fs in os.walk("/kaggle/input"):
        for f in fs:
            if f == "tasks.jsonl":
                return os.path.join(root,f)
    return None

TASKS_PATH = find_tasks()
print("TASKS_PATH:", TASKS_PATH)

MODEL_DIR = None
for root in ["/kaggle/input/models", "/kaggle/input"]:
    if not os.path.isdir(root): continue
    for p in glob.glob(os.path.join(root, "**", "config.json"), recursive=True):
        if "gemma" in p.lower() and "e4b" in p.lower():
            MODEL_DIR = os.path.dirname(p); break
    if MODEL_DIR: break
if MODEL_DIR is None:
    for root in ["/kaggle/input/models", "/kaggle/input"]:
        if not os.path.isdir(root): continue
        for p in glob.glob(os.path.join(root, "**", "config.json"), recursive=True):
            if "gemma" in p.lower():
                MODEL_DIR = os.path.dirname(p); break
        if MODEL_DIR: break
print("MODEL_DIR:", MODEL_DIR)


In [ ]:
src = base64.b64decode("IiIiQnVpbGQgb3JhY2xlLWd1aWRlZCBTRlQgdHJhamVjdG9yaWVzIGZyb20gdGhlIGNvbXBldGl0aW9uJ3MgcHVibGljIHRhc2tzLgoKUnVucyAxMDAlIG9mZmxpbmUgKG5vIGV4dGVybmFsIEFQSXMpLiBFbWl0cyBvbmUgSlNPTiBvYmplY3QgcGVyIHRhc2sgd2l0aCBhbgpPcGVuQUktc3R5bGUgYGBtZXNzYWdlc2BgIGxpc3Q7IGFzc2lzdGFudCB0dXJucyBjYXJyeSB0aGUgdG9vbCBjYWxscywgc28gdGhlCnRyYWluaW5nIGtlcm5lbCBjYW4gYXBwbHkgdGhlIG9mZmljaWFsIGdlbW1hLTQtZTRiLWl0IGNoYXQgdGVtcGxhdGUgYW5kIG1hc2sKbGFiZWxzIHRvIGFzc2lzdGFudCB0b2tlbnMgb25seSAobGFiZWxzPS0xMDAgZWxzZXdoZXJlKS4KIiIiCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBhcmdwYXJzZQppbXBvcnQganNvbgppbXBvcnQgb3MKaW1wb3J0IHJhbmRvbQppbXBvcnQgcmUKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgpTVFJBVEFfS0VZUyA9ICgiZmFzdGFwaSIsICJyaWNoIiwgInJlcXVlc3RzIiwgImh0dHB4IikKCgpkZWYgZmluZF90YXNrcyhleHBsaWNpdD1Ob25lKToKICAgIGNhbmRpZGF0ZXMgPSBbXQogICAgaWYgZXhwbGljaXQ6CiAgICAgICAgY2FuZGlkYXRlcy5hcHBlbmQoUGF0aChleHBsaWNpdCkpCiAgICBlbnYgPSBvcy5lbnZpcm9uLmdldCgiVEFTS1NfSlNPTkwiKQogICAgaWYgZW52OgogICAgICAgIGNhbmRpZGF0ZXMuYXBwZW5kKFBhdGgoZW52KSkKICAgIGNhbmRpZGF0ZXMgKz0gWwogICAgICAgIFBhdGgoInB1Ymxpc2hlZC90YXNrcy5qc29ubCIpLAogICAgICAgIFBhdGgoIi9rYWdnbGUvaW5wdXQvY29tcGV0aXRpb25zL2dlbW1hLTQtZGV2ZWxvcGVyLWFnZW50L3Rhc2tzLmpzb25sIiksCiAgICAgICAgUGF0aCgiL2thZ2dsZS9pbnB1dC9nZW1tYS00LWRldmVsb3Blci1hZ2VudC90YXNrcy5qc29ubCIpLAogICAgICAgIFBhdGgoIi90bXAvcHVibGlzaGVkL3Rhc2tzLmpzb25sIiksCiAgICBdCiAgICBmb3IgYyBpbiBjYW5kaWRhdGVzOgogICAgICAgIGlmIGMgYW5kIGMuaXNfZmlsZSgpOgogICAgICAgICAgICByZXR1cm4gYwogICAgcmV0dXJuIE5vbmUKCgpkZWYgc3RyYXR1bV9vZih0YXNrKToKICAgIHJlcG8gPSAodGFzay5nZXQoInJlcG8iKSBvciB0YXNrLmdldCgicmVwb3NpdG9yeSIpIG9yICIiKS5sb3dlcigpCiAgICBmb3IgayBpbiBTVFJBVEFfS0VZUzoKICAgICAgICBpZiBrIGluIHJlcG86CiAgICAgICAgICAgIHJldHVybiBrCiAgICAjIGZhbGxiYWNrOiBpbmZlciBmcm9tIHRhc2sgaWQgLyBwcm9ibGVtIHN0YXRlbWVudAogICAgdGV4dCA9ICh0YXNrLmdldCgicHJvYmxlbV9zdGF0ZW1lbnQiKSBvciB0YXNrLmdldCgiaXNzdWUiKSBvciAiIikubG93ZXIoKQogICAgZm9yIGsgaW4gU1RSQVRBX0tFWVM6CiAgICAgICAgaWYgayBpbiB0ZXh0OgogICAgICAgICAgICByZXR1cm4gawogICAgcmV0dXJuICJvdGhlciIKCgpkZWYgbWFrZV9zcGxpdCh0YXNrcywgdHJhaW5fbj0xMDAsIHZhbF9uPTI5LCBzZWVkPTQyKToKICAgIHJuZyA9IHJhbmRvbS5SYW5kb20oc2VlZCkKICAgIGdyb3VwcyA9IHt9CiAgICBmb3IgdCBpbiB0YXNrczoKICAgICAgICBncm91cHMuc2V0ZGVmYXVsdChzdHJhdHVtX29mKHQpLCBbXSkuYXBwZW5kKHQpCiAgICB0cmFpbiwgdmFsID0gW10sIFtdCiAgICBmb3IgZywgaXRlbXMgaW4gZ3JvdXBzLml0ZW1zKCk6CiAgICAgICAgaXRlbXMgPSBzb3J0ZWQoaXRlbXMsIGtleT1sYW1iZGEgeDogeC5nZXQoImlkIikgb3IgeC5nZXQoInRhc2tfaWQiKSBvciAiIikKICAgICAgICBybmcuc2h1ZmZsZShpdGVtcykKICAgICAgICAjIHByb3BvcnRpb25hbCBhbGxvY2F0aW9uCiAgICAgICAgbiA9IGxlbihpdGVtcykKICAgICAgICBuX3RyYWluID0gcm91bmQobiAqIHRyYWluX24gLyAodHJhaW5fbiArIHZhbF9uKSkKICAgICAgICBuX3RyYWluID0gbWluKG5fdHJhaW4sIG4pCiAgICAgICAgdHJhaW4uZXh0ZW5kKGl0ZW1zWzpuX3RyYWluXSkKICAgICAgICB2YWwuZXh0ZW5kKGl0ZW1zW25fdHJhaW46XSkKICAgICMgYWRqdXN0IHRvIGV4YWN0IGNvdW50cyBpZiByb3VuZGluZyBkcmlmdGVkCiAgICBybmcuc2h1ZmZsZSh0cmFpbikKICAgIHJuZy5zaHVmZmxlKHZhbCkKICAgIHdoaWxlIGxlbih0cmFpbikgPiB0cmFpbl9uIGFuZCB2YWw6CiAgICAgICAgdmFsLmluc2VydCgwLCB0cmFpbi5wb3AoKSkKICAgIHdoaWxlIGxlbih2YWwpID4gdmFsX24gYW5kIHRyYWluOgogICAgICAgIHRyYWluLmluc2VydCgwLCB2YWwucG9wKCkpCiAgICAjIGlmIG5vdCBlbm91Z2gsIHBhZCBmcm9tIHRoZSBsYXJnZXIKICAgIHdoaWxlIGxlbih0cmFpbikgPCB0cmFpbl9uIGFuZCBsZW4odmFsKSA+IDA6CiAgICAgICAgdHJhaW4uYXBwZW5kKHZhbC5wb3AoMCkpCiAgICB3aGlsZSBsZW4odmFsKSA8IHZhbF9uIGFuZCBsZW4odHJhaW4pID4gdHJhaW5fbjoKICAgICAgICB2YWwuYXBwZW5kKHRyYWluLnBvcCgpKQogICAgcmV0dXJuIHRyYWluLCB2YWwKCgpkZWYgc2F2ZV9zcGxpdChwYXRoLCB0cmFpbiwgdmFsKToKICAgIG9iaiA9IHsKICAgICAgICAidHJhaW5faWRzIjogW3QuZ2V0KCJpZCIpIG9yIHQuZ2V0KCJ0YXNrX2lkIikgZm9yIHQgaW4gdHJhaW5dLAogICAgICAgICJ2YWxfaWRzIjogW3QuZ2V0KCJpZCIpIG9yIHQuZ2V0KCJ0YXNrX2lkIikgZm9yIHQgaW4gdmFsXSwKICAgICAgICAidHJhaW5fY291bnQiOiBsZW4odHJhaW4pLAogICAgICAgICJ2YWxfY291bnQiOiBsZW4odmFsKSwKICAgICAgICAic3RyYXRhIjoge30sCiAgICB9CiAgICBmb3IgdCBpbiB0cmFpbjoKICAgICAgICBvYmpbInN0cmF0YSJdLnNldGRlZmF1bHQoc3RyYXR1bV9vZih0KSwgeyJ0cmFpbiI6IDAsICJ2YWwiOiAwfSkKICAgICAgICBvYmpbInN0cmF0YSJdW3N0cmF0dW1fb2YodCldWyJ0cmFpbiJdICs9IDEKICAgIGZvciB0IGluIHZhbDoKICAgICAgICBvYmpbInN0cmF0YSJdLnNldGRlZmF1bHQoc3RyYXR1bV9vZih0KSwgeyJ0cmFpbiI6IDAsICJ2YWwiOiAwfSkKICAgICAgICBvYmpbInN0cmF0YSJdW3N0cmF0dW1fb2YodCldWyJ2YWwiXSArPSAxCiAgICBQYXRoKHBhdGgpLndyaXRlX3RleHQoanNvbi5kdW1wcyhvYmosIGluZGVudD0yLCBlbnN1cmVfYXNjaWk9RmFsc2UpLCBlbmNvZGluZz0idXRmLTgiKQogICAgcmV0dXJuIG9iagoKCmRlZiBwYXJzZV9wYXRjaChwYXRjaCk6CiAgICBmaWxlcyA9IFtdCiAgICBjdXIgPSBOb25lCiAgICBmb3IgbGluZSBpbiAocGF0Y2ggb3IgIiIpLnNwbGl0bGluZXMoKToKICAgICAgICBpZiBsaW5lLnN0YXJ0c3dpdGgoIi0tLSAiKToKICAgICAgICAgICAgcCA9IGxpbmVbNDpdLnN0cmlwKCkKICAgICAgICAgICAgaWYgcC5zdGFydHN3aXRoKCJhLyIpOgogICAgICAgICAgICAgICAgcCA9IHBbMjpdCiAgICAgICAgICAgIGlmIHAgPT0gIi9kZXYvbnVsbCI6CiAgICAgICAgICAgICAgICBwID0gTm9uZQogICAgICAgICAgICBjdXIgPSB7InBhdGgiOiBwLCAiaHVua3MiOiBbXX0KICAgICAgICAgICAgZmlsZXMuYXBwZW5kKGN1cikKICAgICAgICBlbGlmIGxpbmUuc3RhcnRzd2l0aCgiQEAiKSBhbmQgY3VyIGlzIG5vdCBOb25lOgogICAgICAgICAgICBtID0gcmUubWF0Y2gociJAQCAtKFxkKykoPzosXGQrKT8gXCsoXGQrKSg/OixcZCspPyBAQCguKikiLCBsaW5lKQogICAgICAgICAgICBpZiBtOgogICAgICAgICAgICAgICAgY3VyWyJodW5rcyJdLmFwcGVuZCh7Im9sZF9zdGFydCI6IGludChtLmdyb3VwKDEpKSwgIm5ld19zdGFydCI6IGludChtLmdyb3VwKDIpKSwgImxpbmVzIjogW119KQogICAgICAgIGVsaWYgY3VyIGlzIG5vdCBOb25lIGFuZCBjdXJbImh1bmtzIl06CiAgICAgICAgICAgIGlmIGxpbmUuc3RhcnRzd2l0aCgiKyIpIGFuZCBub3QgbGluZS5zdGFydHN3aXRoKCIrKysiKToKICAgICAgICAgICAgICAgIGN1clsiaHVua3MiXVstMV1bImxpbmVzIl0uYXBwZW5kKCgiYWRkIiwgbGluZVsxOl0pKQogICAgICAgICAgICBlbGlmIGxpbmUuc3RhcnRzd2l0aCgiLSIpIGFuZCBub3QgbGluZS5zdGFydHN3aXRoKCItLS0iKToKICAgICAgICAgICAgICAgIGN1clsiaHVua3MiXVstMV1bImxpbmVzIl0uYXBwZW5kKCgiZGVsIiwgbGluZVsxOl0pKQogICAgICAgICAgICBlbGlmIGxpbmUuc3RhcnRzd2l0aCgiICIpOgogICAgICAgICAgICAgICAgY3VyWyJodW5rcyJdWy0xXVsibGluZXMiXS5hcHBlbmQoKCJjdHgiLCBsaW5lWzE6XSkpCiAgICAgICAgICAgIGVsaWYgbGluZS5zdGFydHN3aXRoKCJcXCIpOgogICAgICAgICAgICAgICAgcGFzcwogICAgIyBkcm9wIE5vbmUgcGF0aHMKICAgIHJldHVybiBbZiBmb3IgZiBpbiBmaWxlcyBpZiBmWyJwYXRoIl1dCgoKZGVmIGV4dHJhY3Rfc3ltYm9scyhwYXRjaCk6CiAgICBzeW1zID0gW10KICAgIGZvciBtIGluIHJlLmZpbmRpdGVyKHIiXlsrLV0uKlxiKD86ZGVmfGNsYXNzKVxzKyhbQS1aYS16X11bQS1aYS16MC05X10qKSIsIHBhdGNoIG9yICIiLCByZS5NKToKICAgICAgICBzID0gbS5ncm91cCgxKQogICAgICAgIGlmIHMgbm90IGluIHN5bXM6CiAgICAgICAgICAgIHN5bXMuYXBwZW5kKHMpCiAgICByZXR1cm4gc3ltcwoKCmRlZiBodW5rX3RleHRzKGh1bmspOgogICAgb2xkID0gW3QgZm9yIGtpbmQsIHQgaW4gaHVua1sibGluZXMiXSBpZiBraW5kIGluICgiY3R4IiwgImRlbCIpXQogICAgbmV3ID0gW3QgZm9yIGtpbmQsIHQgaW4gaHVua1sibGluZXMiXSBpZiBraW5kIGluICgiY3R4IiwgImFkZCIpXQogICAgcmV0dXJuICJcbiIuam9pbihvbGQpLCAiXG4iLmpvaW4obmV3KQoKCmRlZiBidWlsZF90cmFqZWN0b3J5KHRhc2spOgogICAgdGlkID0gdGFzay5nZXQoImlkIikgb3IgdGFzay5nZXQoInRhc2tfaWQiKSBvciAidGFzayIKICAgIHByb2JsZW0gPSB0YXNrLmdldCgicHJvYmxlbV9zdGF0ZW1lbnQiKSBvciB0YXNrLmdldCgiaXNzdWUiKSBvciB0YXNrLmdldCgiZGVzY3JpcHRpb24iKSBvciAiIgogICAgcGF0Y2ggPSB0YXNrLmdldCgicGF0Y2giKSBvciB0YXNrLmdldCgiZ29sZF9wYXRjaCIpIG9yICIiCiAgICBmaWxlcyA9IHBhcnNlX3BhdGNoKHBhdGNoKQogICAgIyBwaWNrIGZpcnN0IGNoYW5nZWQgZmlsZSB3aXRoIGEgaHVuawogICAgY2hvc2VuID0gTm9uZQogICAgZm9yIGYgaW4gZmlsZXM6CiAgICAgICAgaWYgZlsiaHVua3MiXToKICAgICAgICAgICAgY2hvc2VuID0gZgogICAgICAgICAgICBicmVhawogICAgaWYgY2hvc2VuIGlzIE5vbmU6CiAgICAgICAgY2hvc2VuID0geyJwYXRoIjogInNyYy9tYWluLnB5IiwgImh1bmtzIjogW3sib2xkX3N0YXJ0IjogMSwgIm5ld19zdGFydCI6IDEsICJsaW5lcyI6IFtdfV19CiAgICBmcGF0aCA9IGNob3NlblsicGF0aCJdCiAgICBodW5rID0gY2hvc2VuWyJodW5rcyJdWzBdCiAgICBMID0gbWF4KDEsIGh1bmtbIm9sZF9zdGFydCJdKQogICAgb2xkX3RleHQsIG5ld190ZXh0ID0gaHVua190ZXh0cyhodW5rKQogICAgaWYgbm90IG9sZF90ZXh0LnN0cmlwKCk6CiAgICAgICAgb2xkX3RleHQgPSAiIyAobm8gY29udGV4dCkiCiAgICBpZiBub3QgbmV3X3RleHQuc3RyaXAoKToKICAgICAgICBuZXdfdGV4dCA9IG9sZF90ZXh0CiAgICBzeW1zID0gZXh0cmFjdF9zeW1ib2xzKHBhdGNoKQogICAgc3ltID0gc3ltc1swXSBpZiBzeW1zIGVsc2UgKHByb2JsZW0uc3BsaXQoKVs6MV0gb3IgWyJidWciXSlbMF0KICAgIHN5bSA9IHJlLnN1YihyIlteQS1aYS16MC05X10iLCAiIiwgc3ltKSBvciAidGFyZ2V0IgogICAgZml4X2Rlc2MgPSBmImZpeCB7c3ltfSBpbiB7ZnBhdGh9IgoKICAgIHJlYWRfc3RhcnQgPSBtYXgoMSwgTCAtIDIwKQogICAgcmVhZF9lbmQgPSBMICsgMzUKCiAgICBzdGF0ZSA9IGxhbWJkYSBzdGVwOiBmIlNUQVRFOiB7ZnBhdGh9OntMfSB8IHtmaXhfZGVzY30gfCB7c3RlcH0iCgogICAgbWVzc2FnZXMgPSBbXQogICAgIyBzeXN0ZW0gbWluaW1hbCAob3B0aW9uYWwpCiAgICBtZXNzYWdlcy5hcHBlbmQoeyJyb2xlIjogInN5c3RlbSIsICJjb250ZW50IjogIllvdSBhcmUgYW4gYXV0b25vbW91cyBzb2Z0d2FyZSBlbmdpbmVlci4gTWFrZSB0aGUgc21hbGxlc3QgY29ycmVjdCBjaGFuZ2UuIn0pCiAgICBtZXNzYWdlcy5hcHBlbmQoeyJyb2xlIjogInVzZXIiLCAiY29udGVudCI6IHByb2JsZW19KQoKICAgICMgVHVybiAxIGxvY2FsaXphdGlvbgogICAgbWVzc2FnZXMuYXBwZW5kKHsKICAgICAgICAicm9sZSI6ICJhc3Npc3RhbnQiLAogICAgICAgICJjb250ZW50IjogZiI8dGhpbms+QnVnIGxpa2VseSBpbiB7c3ltfSBpbiB7ZnBhdGh9OyBsb2NhbGl6ZSB3aXRoIGdyZXAuPC90aGluaz5cbntzdGF0ZSgnbG9jYXRlJyl9IiwKICAgICAgICAidG9vbF9jYWxscyI6IFt7InR5cGUiOiAiZnVuY3Rpb24iLCAiZnVuY3Rpb24iOiB7Im5hbWUiOiAicnVuX2NvbW1hbmQiLAogICAgICAgICAgICAiYXJndW1lbnRzIjoganNvbi5kdW1wcyh7ImNvbW1hbmQiOiBmImdpdCAtLW5vLXBhZ2VyIGdyZXAgLW4gXCJ7c3ltfVwiIC0tICc6IXRlc3RzLyonIn0pfX1dLAogICAgfSkKICAgIG1lc3NhZ2VzLmFwcGVuZCh7InJvbGUiOiAidG9vbCIsICJjb250ZW50IjogZiJ7ZnBhdGh9OntMfToge3N5bX0ifSkKCiAgICAjIFR1cm4gMiBzdXJnaWNhbCByZWFkCiAgICBtZXNzYWdlcy5hcHBlbmQoewogICAgICAgICJyb2xlIjogImFzc2lzdGFudCIsCiAgICAgICAgImNvbnRlbnQiOiBmIjx0aGluaz5SZWFkIHRoZSBleGFjdCBodW5rIGJlZm9yZSBlZGl0aW5nLjwvdGhpbms+XG57c3RhdGUoJ2xvY2F0ZScpfSIsCiAgICAgICAgInRvb2xfY2FsbHMiOiBbeyJ0eXBlIjogImZ1bmN0aW9uIiwgImZ1bmN0aW9uIjogeyJuYW1lIjogInJlYWRfZmlsZSIsCiAgICAgICAgICAgICJhcmd1bWVudHMiOiBqc29uLmR1bXBzKHsiZmlsZV9wYXRoIjogZnBhdGgsICJzdGFydF9saW5lIjogcmVhZF9zdGFydCwgImVuZF9saW5lIjogcmVhZF9lbmR9KX19XSwKICAgIH0pCiAgICBtZXNzYWdlcy5hcHBlbmQoeyJyb2xlIjogInRvb2wiLCAiY29udGVudCI6IG9sZF90ZXh0fSkKCiAgICAjIFR1cm4gMyBleGFjdCBlZGl0CiAgICBtZXNzYWdlcy5hcHBlbmQoewogICAgICAgICJyb2xlIjogImFzc2lzdGFudCIsCiAgICAgICAgImNvbnRlbnQiOiBmIjx0aGluaz5BcHBseSB0aGUgZ29sZCBmaXg6IHJlcGxhY2UgdGhlIGJ1Z2d5IGh1bmsgd2l0aCB0aGUgY29ycmVjdGVkIHZlcnNpb24uPC90aGluaz5cbntzdGF0ZSgnZWRpdCcpfSIsCiAgICAgICAgInRvb2xfY2FsbHMiOiBbeyJ0eXBlIjogImZ1bmN0aW9uIiwgImZ1bmN0aW9uIjogeyJuYW1lIjogImVkaXRfZmlsZSIsCiAgICAgICAgICAgICJhcmd1bWVudHMiOiBqc29uLmR1bXBzKHsiZmlsZV9wYXRoIjogZnBhdGgsICJvbGRfc3RyaW5nIjogb2xkX3RleHQsICJuZXdfc3RyaW5nIjogbmV3X3RleHR9KX19XSwKICAgIH0pCiAgICBtZXNzYWdlcy5hcHBlbmQoeyJyb2xlIjogInRvb2wiLCAiY29udGVudCI6ICJTdWNjZXNzZnVsbHkgZWRpdGVkIGZpbGUuIn0pCgogICAgIyBUdXJuIDQgdmVyaWZ5ICsgc3VibWl0CiAgICBtZXNzYWdlcy5hcHBlbmQoewogICAgICAgICJyb2xlIjogImFzc2lzdGFudCIsCiAgICAgICAgImNvbnRlbnQiOiBmIjx0aGluaz5TeW50YXgtY2hlY2sgYW5kIHJ1biB0YXJnZXRlZCB0ZXN0cy48L3RoaW5rPlxue3N0YXRlKCd2ZXJpZnknKX0iLAogICAgICAgICJ0b29sX2NhbGxzIjogW3sidHlwZSI6ICJmdW5jdGlvbiIsICJmdW5jdGlvbiI6IHsibmFtZSI6ICJydW5fY29tbWFuZCIsCiAgICAgICAgICAgICJhcmd1bWVudHMiOiBqc29uLmR1bXBzKHsiY29tbWFuZCI6IGYicHl0aG9uMyAtbSBweV9jb21waWxlIHtmcGF0aH0gJiYgdGltZW91dCA2MCBweXRlc3QgLXEgMj4mMSB8IHRhaWwgLW4gODAifSl9fV0sCiAgICB9KQogICAgbWVzc2FnZXMuYXBwZW5kKHsicm9sZSI6ICJ0b29sIiwgImNvbnRlbnQiOiAicHlfY29tcGlsZSBvazsgdGVzdHMgcGFzc2VkIn0pCgogICAgbWVzc2FnZXMuYXBwZW5kKHsKICAgICAgICAicm9sZSI6ICJhc3Npc3RhbnQiLAogICAgICAgICJjb250ZW50IjogZiI8dGhpbms+UHJlLXN1Ym1pdCBpbnRlZ3JpdHkgY2hlY2suPC90aGluaz5cbntzdGF0ZSgndmVyaWZ5Jyl9IiwKICAgICAgICAidG9vbF9jYWxscyI6IFt7InR5cGUiOiAiZnVuY3Rpb24iLCAiZnVuY3Rpb24iOiB7Im5hbWUiOiAicnVuX2NvbW1hbmQiLAogICAgICAgICAgICAiYXJndW1lbnRzIjoganNvbi5kdW1wcyh7ImNvbW1hbmQiOiAiZ2l0IGNoZWNrb3V0IC0tIHRlc3RzLyB0ZXN0LyB0ZXN0aW5nLyBjb25mdGVzdC5weSAyPi9kZXYvbnVsbDsgZ2l0IHN0YXR1cyAtLXNob3J0ICYmIGdpdCAtLW5vLXBhZ2VyIGRpZmYgLS1zdGF0In0pfX1dLAogICAgfSkKICAgIG1lc3NhZ2VzLmFwcGVuZCh7InJvbGUiOiAidG9vbCIsICJjb250ZW50IjogZiJNIHtmcGF0aH0ifSkKCiAgICBtZXNzYWdlcy5hcHBlbmQoewogICAgICAgICJyb2xlIjogImFzc2lzdGFudCIsCiAgICAgICAgImNvbnRlbnQiOiBmIjx0aGluaz5TdWJtaXQgdGhlIHBhdGNoLjwvdGhpbms+XG57c3RhdGUoJ3N1Ym1pdCcpfSIsCiAgICAgICAgInRvb2xfY2FsbHMiOiBbeyJ0eXBlIjogImZ1bmN0aW9uIiwgImZ1bmN0aW9uIjogeyJuYW1lIjogInN1Ym1pdF9wYXRjaCIsICJhcmd1bWVudHMiOiBqc29uLmR1bXBzKHt9KX19XSwKICAgIH0pCiAgICBtZXNzYWdlcy5hcHBlbmQoeyJyb2xlIjogInRvb2wiLCAiY29udGVudCI6ICJwYXRjaF9zaXplID4gMCwgZmlsZXNfY2hhbmdlZDogMSJ9KQoKICAgIHJldHVybiB7InRhc2tfaWQiOiB0aWQsICJtZXNzYWdlcyI6IG1lc3NhZ2VzfQoKCmRlZiB0cnlfdG9rZW5pemUocmVjb3JkcywgdG9rZW5pemVyX25hbWUpOgogICAgdHJ5OgogICAgICAgIGZyb20gdHJhbnNmb3JtZXJzIGltcG9ydCBBdXRvVG9rZW5pemVyCiAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGU6CiAgICAgICAgcHJpbnQoZiJbdG9rZW5pemVdIHRyYW5zZm9ybWVycyB1bmF2YWlsYWJsZToge2V9IikKICAgICAgICByZXR1cm4gcmVjb3JkcwogICAgdHJ5OgogICAgICAgIHRvayA9IEF1dG9Ub2tlbml6ZXIuZnJvbV9wcmV0cmFpbmVkKHRva2VuaXplcl9uYW1lKQogICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBlOgogICAgICAgIHByaW50KGYiW3Rva2VuaXplXSBjb3VsZCBub3QgbG9hZCB0b2tlbml6ZXIge3Rva2VuaXplcl9uYW1lfToge2V9IikKICAgICAgICByZXR1cm4gcmVjb3JkcwogICAgb3V0ID0gW10KICAgIGZvciByIGluIHJlY29yZHM6CiAgICAgICAgdHJ5OgogICAgICAgICAgICByZW5kZXJlZCA9IHRvay5hcHBseV9jaGF0X3RlbXBsYXRlKAogICAgICAgICAgICAgICAgclsibWVzc2FnZXMiXSwgdG9rZW5pemU9VHJ1ZSwgcmV0dXJuX2RpY3Q9VHJ1ZSwKICAgICAgICAgICAgICAgIHJldHVybl9hc3Npc3RhbnRfdG9rZW5zX21hc2s9VHJ1ZSwgYWRkX2dlbmVyYXRpb25fcHJvbXB0PUZhbHNlLAogICAgICAgICAgICApCiAgICAgICAgICAgIGlkcyA9IHJlbmRlcmVkWyJpbnB1dF9pZHMiXQogICAgICAgICAgICBtYXNrID0gcmVuZGVyZWQuZ2V0KCJhc3Npc3RhbnRfbWFza3MiKSBvciByZW5kZXJlZC5nZXQoImFzc2lzdGFudF90b2tlbnNfbWFzayIpCiAgICAgICAgICAgIGlmIG1hc2sgaXMgTm9uZToKICAgICAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigibm8gYXNzaXN0YW50IG1hc2sgcmV0dXJuZWQiKQogICAgICAgICAgICBsYWJlbHMgPSBbaW50KHgpIGlmIGludChtKSA9PSAxIGVsc2UgLTEwMCBmb3IgeCwgbSBpbiB6aXAoaWRzLCBtYXNrKV0KICAgICAgICAgICAgciA9IGRpY3QocikKICAgICAgICAgICAgclsiaW5wdXRfaWRzIl0gPSBbaW50KHgpIGZvciB4IGluIGlkc10KICAgICAgICAgICAgclsibGFiZWxzIl0gPSBsYWJlbHMKICAgICAgICAgICAgclsidG9rZW5pemVyIl0gPSB0b2tlbml6ZXJfbmFtZQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZToKICAgICAgICAgICAgcHJpbnQoZiJbdG9rZW5pemVdIGZhaWxlZCBmb3Ige3JbJ3Rhc2tfaWQnXX06IHtlfSIpCiAgICAgICAgb3V0LmFwcGVuZChyKQogICAgcmV0dXJuIG91dAoKCmRlZiBtYWluKCk6CiAgICBhcCA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKCkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS10YXNrcyIsIGRlZmF1bHQ9Tm9uZSkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1zcGxpdCIsIGRlZmF1bHQ9Tm9uZSwgaGVscD0icGF0aCB0byBzcGxpdCBqc29uIChsb2FkZWQgb3IgY3JlYXRlZCkiKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLW91dCIsIGRlZmF1bHQ9Im9yYWNsZV90cmFqZWN0b3JpZXMuanNvbmwiKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLXRyYWluLW9ubHkiLCBhY3Rpb249InN0b3JlX3RydWUiLCBkZWZhdWx0PVRydWUpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tdG9rZW5pemVyIiwgZGVmYXVsdD0iZ29vZ2xlL2dlbW1hLTQtZTRiLWl0IikKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1zZWVkIiwgdHlwZT1pbnQsIGRlZmF1bHQ9NDIpCiAgICBhcmdzID0gYXAucGFyc2VfYXJncygpCgogICAgdGFza3NfcGF0aCA9IGZpbmRfdGFza3MoYXJncy50YXNrcykKICAgIGlmIHRhc2tzX3BhdGggaXMgTm9uZToKICAgICAgICByYWlzZSBTeXN0ZW1FeGl0KCJ0YXNrcy5qc29ubCBub3QgZm91bmQ7IHBhc3MgLS10YXNrcyBvciBzZXQgVEFTS1NfSlNPTkwiKQogICAgdGFza3MgPSBbanNvbi5sb2Fkcyh4KSBmb3IgeCBpbiB0YXNrc19wYXRoLnJlYWRfdGV4dChlbmNvZGluZz0idXRmLTgiKS5zcGxpdGxpbmVzKCkgaWYgeC5zdHJpcCgpXQogICAgcHJpbnQoZiJsb2FkZWQge2xlbih0YXNrcyl9IHRhc2tzIGZyb20ge3Rhc2tzX3BhdGh9IikKCiAgICAjIHNwbGl0CiAgICBzcGxpdF9wYXRoID0gUGF0aChhcmdzLnNwbGl0KSBpZiBhcmdzLnNwbGl0IGVsc2UgUGF0aChfX2ZpbGVfXykucmVzb2x2ZSgpLnBhcmVudCAvICJzcGxpdF8xMDBfMjkuanNvbiIKICAgIGlmIHNwbGl0X3BhdGguaXNfZmlsZSgpOgogICAgICAgIHRyeToKICAgICAgICAgICAgc3AgPSBqc29uLmxvYWRzKHNwbGl0X3BhdGgucmVhZF90ZXh0KGVuY29kaW5nPSJ1dGYtOCIpKQogICAgICAgICAgICBpZHNldCA9IHNldChzcC5nZXQoInRyYWluX2lkcyIsIFtdKSkKICAgICAgICAgICAgaWYgaWRzZXQ6CiAgICAgICAgICAgICAgICB0cmFpbl90YXNrcyA9IFt0IGZvciB0IGluIHRhc2tzIGlmICh0LmdldCgiaWQiKSBvciB0LmdldCgidGFza19pZCIpKSBpbiBpZHNldF0KICAgICAgICAgICAgZWxzZToKICAgICAgICAgICAgICAgIHRyYWluX3Rhc2tzID0gTm9uZQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgICAgIHRyYWluX3Rhc2tzID0gTm9uZQogICAgZWxzZToKICAgICAgICB0cmFpbl90YXNrcyA9IE5vbmUKCiAgICBpZiB0cmFpbl90YXNrcyBpcyBOb25lOgogICAgICAgIHRyYWluLCB2YWwgPSBtYWtlX3NwbGl0KHRhc2tzLCAxMDAsIDI5LCBhcmdzLnNlZWQpCiAgICAgICAgc2F2ZV9zcGxpdChzcGxpdF9wYXRoLCB0cmFpbiwgdmFsKQogICAgICAgIHRyYWluX3Rhc2tzID0gdHJhaW4KICAgICAgICBwcmludChmImNyZWF0ZWQgc3BsaXQ6IHtsZW4odHJhaW4pfSB0cmFpbiAvIHtsZW4odmFsKX0gdmFsIC0+IHtzcGxpdF9wYXRofSIpCgogICAgcHJpbnQoZiJidWlsZGluZyB0cmFqZWN0b3JpZXMgZm9yIHtsZW4odHJhaW5fdGFza3MpfSB0cmFpbiB0YXNrcyIpCiAgICByZWNvcmRzID0gW2J1aWxkX3RyYWplY3RvcnkodCkgZm9yIHQgaW4gdHJhaW5fdGFza3NdCgogICAgcmVjb3JkcyA9IHRyeV90b2tlbml6ZShyZWNvcmRzLCBhcmdzLnRva2VuaXplcikKCiAgICBvdXRfcGF0aCA9IFBhdGgoYXJncy5vdXQpCiAgICB3aXRoIG91dF9wYXRoLm9wZW4oInciLCBlbmNvZGluZz0idXRmLTgiKSBhcyBmOgogICAgICAgIGZvciByIGluIHJlY29yZHM6CiAgICAgICAgICAgIGYud3JpdGUoanNvbi5kdW1wcyhyLCBlbnN1cmVfYXNjaWk9RmFsc2UpICsgIlxuIikKICAgIHByaW50KGYid3JvdGUge2xlbihyZWNvcmRzKX0gdHJhamVjdG9yaWVzIHRvIHtvdXRfcGF0aH0iKQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBtYWluKCkK").decode("utf-8")
with open("/kaggle/working/build_oracle_trajectories.py","w",encoding="utf-8") as f:
    f.write(src)
import sys, importlib.util
spec = importlib.util.spec_from_file_location("bot","/kaggle/working/build_oracle_trajectories.py")
bot = importlib.util.module_from_spec(spec)
spec.loader.exec_module(bot)
print("loaded bot")


In [ ]:
OUT_JSONL = "/kaggle/working/oracle_trajectories.jsonl"
SPLIT_JSON = "/kaggle/working/split_100_29.json"
if not os.path.isfile(OUT_JSONL):
    if TASKS_PATH is None:
        raise SystemExit("tasks.jsonl not found")
    import sys as _s
    _s.argv = ["build_oracle_trajectories.py", "--tasks", TASKS_PATH, "--split", SPLIT_JSON, "--out", OUT_JSONL, "--tokenizer", MODEL_DIR or "google/gemma-4-e4b-it"]
    bot.main()
print("trajectories:", OUT_JSONL, os.path.getsize(OUT_JSONL) if os.path.isfile(OUT_JSONL) else None)


In [ ]:
records = [json.loads(x) for x in open(OUT_JSONL, encoding="utf-8") if x.strip()]
print("num train trajectories:", len(records))
if os.path.isfile(SPLIT_JSON):
    sp = json.load(open(SPLIT_JSON))
    print("split counts:", sp.get("train_count"), sp.get("val_count"))


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_DIR, torch_dtype=torch.bfloat16, device_map={"":0})
model.config.use_cache = False
model.gradient_checkpointing_enable()
import peft.import_utils as _iu
import peft.tuners.lora.torchao as _ta
_ta.is_torchao_available = lambda: False
lora_config = LoraConfig(
    r=8, lora_alpha=16, target_modules="all-linear",
    lora_dropout=0.05, bias="none", task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


In [ ]:
MAX_SEQ = 1024

def tokenize_record(rec):
    msgs = rec["messages"]
    _out = tokenizer.apply_chat_template(msgs, tokenize=True, add_generation_prompt=False, return_dict=True)
    full_ids = _out["input_ids"] if isinstance(_out, dict) else _out.input_ids
    if hasattr(full_ids, "tolist"): full_ids = full_ids.tolist()
    if isinstance(full_ids, list) and full_ids and isinstance(full_ids[0], list): full_ids = full_ids[0]
    labels = [-100] * len(full_ids)
    cur = 0
    for i, m in enumerate(msgs):
        _p = tokenizer.apply_chat_template(msgs[:i+1], tokenize=True, add_generation_prompt=False, return_dict=True)
        prefix_ids = _p["input_ids"] if isinstance(_p, dict) else _p.input_ids
        if hasattr(prefix_ids, "tolist"): prefix_ids = prefix_ids.tolist()
        if isinstance(prefix_ids, list) and prefix_ids and isinstance(prefix_ids[0], list): prefix_ids = prefix_ids[0]
        start = cur
        end = min(len(prefix_ids), len(full_ids))
        if m.get("role") == "assistant":
            for j in range(start, end):
                labels[j] = full_ids[j]
        cur = end
    if len(full_ids) > MAX_SEQ:
        full_ids = full_ids[:MAX_SEQ]
        labels = labels[:MAX_SEQ]
    return {"input_ids": full_ids, "labels": labels, "attention_mask": [1]*len(full_ids)}

tok_records = []
for r in records:
    try:
        tok_records.append(tokenize_record(r))
    except Exception as e:
        print("tokenize fail", r.get("task_id"), e)
print("tokenized:", len(tok_records))


In [ ]:
from torch.utils.data import Dataset
class TrajDataset(Dataset):
    def __init__(self, recs):
        self.recs = recs
    def __len__(self):
        return len(self.recs)
    def __getitem__(self, i):
        r = self.recs[i]
        return {"input_ids": torch.tensor(r["input_ids"], dtype=torch.long),
                "labels": torch.tensor(r["labels"], dtype=torch.long),
                "attention_mask": torch.tensor(r["attention_mask"], dtype=torch.long)}
dataset = TrajDataset(tok_records)
print("dataset size:", len(dataset))


In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq

collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, padding=True, label_pad_token_id=-100)

args = TrainingArguments(
    output_dir="/kaggle/working/adapters/main_lora",
    num_train_epochs=2,
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    bf16=True,
    logging_steps=5,
    save_strategy="epoch",
    report_to="none",
    remove_unused_columns=False,
    remove_unused_columns=False,
    gradient_checkpointing=True,
)
trainer = Trainer(model=model, args=args, train_dataset=dataset, data_collator=collator)
trainer.train()
trainer.save_model("/kaggle/working/adapters/main_lora")
tokenizer.save_pretrained("/kaggle/working/adapters/main_lora")
print("saved adapter")


In [ ]:
import json as _j
metrics = {
    "train_trajectories": len(records),
    "tokenized": len(tok_records),
    "epochs": 2,
    "lora_r": 8,
    "lora_alpha": 16,
    "max_seq": MAX_SEQ,
}
open("/kaggle/working/lora_metrics.json","w").write(_j.dumps(metrics, indent=2))
print(_j.dumps(metrics, indent=2))

import os
ad = "/kaggle/working/adapters/main_lora"
print("adapter files:", os.listdir(ad) if os.path.isdir(ad) else "MISSING")
